# Week 4 — CASE WHEN: Conditional Logic in SQL
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Every column you have queried so far already existed in the database. Today you build columns that *don't* — categories that live only as a rule the business applies to its data. `orders` stores eight raw `order_status` values; nowhere does it store "Completed" or "In Progress". `CASE WHEN` is how you write that rule in SQL, name the result, and then group by the category you just invented.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom. The check cells read the exact column aliases each question asks for, so use the alias names as written.

🤖 **Using DeepSeek this week:** you may ask DeepSeek to help draft these queries, but the prompt-then-verify protocol from the demo still applies — tell it the table, the exact columns and that the dialect is SQLite, then **run the query and check the number against the Expected value before you trust it**. The check cells are the verification step; never edit one to make a wrong query pass, and be ready to explain every branch of a `CASE` you submit.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 10.2 MB/s eta 0:00:00


## Question 1 — Map raw statuses onto business categories

Operations care about all eight `order_status` values. Management cares about four. Write a query against `orders` that returns **one row per raw status**, showing the status next to the business category it belongs to and how many orders carry it.

Use a `CASE` expression with these branches, in this order:

- `'delivered'` → `'Completed'`
- `'shipped'`, `'invoiced'`, `'processing'`, `'approved'` → `'In Progress'` (one branch, using `IN`)
- `'canceled'` → `'Canceled'`
- everything else → `'Other'`

Return three columns — `order_status`, the `CASE` expression aliased as `status_group`, and `COUNT(*)` aliased as `order_count` — grouped by `order_status` and sorted by `order_count` descending.

**Expected:** `delivered` / Completed **96,478** | `shipped` / In Progress **1,107** | `canceled` / Canceled **625** | `unavailable` / Other **609** | `invoiced` / In Progress **314** | `processing` / In Progress **301**

In [2]:
%%sql q1 <<
-- Your query here
SELECT
    order_status,
    CASE
        WHEN order_status = 'delivered' THEN 'Completed'
        WHEN order_status IN ('shipped', 'invoiced', 'processing', 'approved') THEN 'In Progress'
        WHEN order_status = 'canceled' THEN 'Canceled'
        ELSE 'Other'
    END AS status_group,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;

In [3]:
# --- CHECK Q1 — do not edit ---
for col in ['order_status', 'status_group', 'order_count']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT aliases"
assert str(q1.iloc[0]['order_status']) == 'delivered', \
    "Q1: the first row should be 'delivered' — sort by order_count descending"

rows = {str(r['order_status']): r for _, r in q1.iterrows()}
for status, group, n in [('delivered',   'Completed',   96478),
                         ('shipped',     'In Progress', 1107),
                         ('canceled',    'Canceled',    625),
                         ('unavailable', 'Other',       609),
                         ('invoiced',    'In Progress', 314),
                         ('processing',  'In Progress', 301)]:
    assert status in rows, f"Q1: no row for order_status '{status}' — did you GROUP BY order_status?"
    assert int(rows[status]['order_count']) == n, \
        f"Q1: expected {n:,} orders with status '{status}', got {int(rows[status]['order_count']):,}"
    assert str(rows[status]['status_group']) == group, \
        f"Q1: '{status}' should be classified as '{group}', got '{rows[status]['status_group']}'"
print("✅ Q1 correct")
q1  # show the result of your query

✅ Q1 correct


,order_status,status_group,order_count
0,delivered,Completed,96478
1,shipped,In Progress,1107
2,canceled,Canceled,625
3,unavailable,Other,609
4,invoiced,In Progress,314
5,processing,In Progress,301
6,created,Other,5
7,approved,In Progress,2


## Question 2 — The board-slide version

Question 1 still returns one row per raw status, which is not what management asked for. Rewrite it so the result has **one row per business category**: keep exactly the same `CASE` expression aliased as `status_group`, drop `order_status` from the `SELECT` list, and `GROUP BY status_group` instead — grouping by the alias of the category you just invented. Alias the count as `order_count` and sort descending.

This is the whole point of `CASE WHEN`: once the invented column has a name, `GROUP BY` treats it exactly like a column that was in the table all along.

**Expected:** four rows — `Completed` **96,478** and `Canceled` **625**, with `In Progress` and `Other` making up the rest. The four categories together must account for all **99,441** orders in the table.

In [4]:
%%sql q2 <<
-- Your query here
SELECT
    CASE
        WHEN order_status = 'delivered' THEN 'Completed'
        WHEN order_status IN ('shipped', 'invoiced', 'processing', 'approved') THEN 'In Progress'
        WHEN order_status = 'canceled' THEN 'Canceled'
        ELSE 'Other'
    END AS status_group,
    COUNT(*) AS order_count
FROM orders
GROUP BY status_group
ORDER BY order_count DESC;

In [5]:
# --- CHECK Q2 — do not edit ---
for col in ['status_group', 'order_count']:
    assert col in q2.columns, f"Q2: missing the '{col}' column — check your SELECT aliases"
assert q2.shape[0] == 4, \
    f"Q2: expected 4 category rows, got {q2.shape[0]} — group by status_group, not order_status"
assert set(q2['status_group']) == {'Completed', 'In Progress', 'Canceled', 'Other'}, \
    f"Q2: expected the categories Completed / In Progress / Canceled / Other, got {sorted(q2['status_group'])}"

groups = {str(r['status_group']): int(r['order_count']) for _, r in q2.iterrows()}
assert groups['Completed'] == 96478, f"Q2: expected 96,478 Completed orders, got {groups['Completed']:,}"
assert groups['Canceled'] == 625, f"Q2: expected 625 Canceled orders, got {groups['Canceled']:,}"
assert sum(groups.values()) == 99441, \
    (f"Q2: the four categories should account for all 99,441 orders, "
     f"got {sum(groups.values()):,} — a missing ELSE drops rows into a NULL bucket")
counts = [int(v) for v in q2['order_count']]
assert counts == sorted(counts, reverse=True), \
    "Q2: rows are not sorted from the largest category down — add ORDER BY order_count DESC"
print("✅ Q2 correct")
q2  # show the result of your query

✅ Q2 correct


,status_group,order_count
0,Completed,96478
1,In Progress,1724
2,Canceled,625
3,Other,614


## Question 3 — Several counts, one row

A `WHERE` clause filters the whole query: one filter, one number. When the business wants three different counts side by side, you need `SUM(CASE WHEN <condition> THEN 1 ELSE 0 END)` — it scores every row 1 or 0 and adds the scores up, so each column carries its own filter.

Write a query on `orders` that returns a **single row** with four columns:

- `completed_orders` — count of orders with status `'delivered'`
- `canceled_orders` — count of orders with status `'canceled'`
- `unavailable_orders` — count of orders with status `'unavailable'`
- `total_orders` — `COUNT(*)` over the whole table

No `WHERE` and no `GROUP BY` — the conditions live inside the `CASE` expressions.

**Expected:** one row — 96,478 | 625 | 609 | 99,441

In [6]:
%%sql q3 <<
-- Your query here
SELECT
    SUM(CASE WHEN order_status = 'delivered'   THEN 1 ELSE 0 END) AS completed_orders,
    SUM(CASE WHEN order_status = 'canceled'    THEN 1 ELSE 0 END) AS canceled_orders,
    SUM(CASE WHEN order_status = 'unavailable' THEN 1 ELSE 0 END) AS unavailable_orders,
    COUNT(*) AS total_orders
FROM orders;

In [7]:
# --- CHECK Q3 — do not edit ---
assert q3.shape[0] == 1, \
    f"Q3: expected a single row, got {q3.shape[0]} — no GROUP BY is needed here"
for col, n in [('completed_orders', 96478), ('canceled_orders', 625),
               ('unavailable_orders', 609), ('total_orders', 99441)]:
    assert col in q3.columns, f"Q3: missing the '{col}' column — check your SELECT aliases"
    assert int(q3.iloc[0][col]) == n, \
        f"Q3: expected {col} = {n:,}, got {int(q3.iloc[0][col]):,}"
print("✅ Q3 correct")
q3  # show the result of your query

✅ Q3 correct


,completed_orders,canceled_orders,unavailable_orders,total_orders
0,96478,625,609,99441


## Question 4 — Bucket a continuous number into bands

Mapping labels onto labels is one use of `CASE`. The more common analytical use is **binning a number**: `order_payments.payment_value` takes tens of thousands of distinct values, and no one can reason about that. Four named bands, they can.

Query `order_payments` and return one row per band with three columns:

- `value_category` — the `CASE` expression, using these labels **exactly** as written: `'Low (< R$50)'` for values under 50, `'Mid (R$50–200)'` for values under 200, `'High (R$200–500)'` for values under 500, and `'Premium (R$500+)'` for everything else
- `payment_count` — `COUNT(*)` (we name it this, not `order_count`, because `order_payments` holds one row per *payment record*, not per order)
- `avg_value` — `ROUND(AVG(payment_value), 2)`

Group by `value_category` and sort by `avg_value` ascending.

Write the bands **narrow first**. Because the first true branch wins, each `WHEN` only needs its upper bound — by the time SQL reaches `< 200`, everything under 50 has already been claimed. Put a wide band above a narrow one and the narrow band silently returns zero rows.

**Expected:** the four bands above, ordered from the lowest `avg_value` up, and their `payment_count` values summing to **103,886** — every row in `order_payments`. (The curriculum does not publish a per-band figure, so the check verifies the labels, the ordering and the total rather than a fixed count.)

In [8]:
%%sql q4 <<
-- Your query here
SELECT
    CASE
        WHEN payment_value < 50  THEN 'Low (< R$50)'
        WHEN payment_value < 200 THEN 'Mid (R$50–200)'
        WHEN payment_value < 500 THEN 'High (R$200–500)'
        ELSE 'Premium (R$500+)'
    END AS value_category,
    COUNT(*) AS payment_count,
    ROUND(AVG(payment_value), 2) AS avg_value
FROM order_payments
GROUP BY value_category
ORDER BY avg_value ASC;

In [9]:
# --- CHECK Q4 — do not edit ---
for col in ['value_category', 'payment_count', 'avg_value']:
    assert col in q4.columns, f"Q4: missing the '{col}' column — check your SELECT aliases"
bands = {'Low (< R$50)', 'Mid (R$50–200)', 'High (R$200–500)', 'Premium (R$500+)'}
extra = set(q4['value_category']) - bands
assert not extra, f"Q4: unexpected band label(s) {sorted(extra)} — copy the four labels exactly as written"
assert int(q4['payment_count'].sum()) == 103886, \
    (f"Q4: the bands should cover all 103,886 payment records, got "
     f"{int(q4['payment_count'].sum()):,} — a missing ELSE leaves rows in a NULL bucket")
avgs = [float(v) for v in q4['avg_value']]
assert avgs == sorted(avgs), \
    "Q4: rows are not sorted from the lowest avg_value up — add ORDER BY avg_value"
assert q4['value_category'].nunique() == q4.shape[0], \
    "Q4: each band should appear once — group by value_category"
print("✅ Q4 correct")
q4  # show the result of your query

✅ Q4 correct


,value_category,payment_count,avg_value
0,Low (< R$50),21078,32.94
1,Mid (R$50–200),62680,107.98
2,High (R$200–500),15865,289.97
3,Premium (R$500+),4263,925.57


## Question 5 — Turning counts into shares (mind the integer division)

Counts answer "how many". Management asks "what share". Extend Question 3: return a **single row** with five columns —

- `completed_orders` — count of `'delivered'` orders, via `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`
- `canceled_orders` — count of `'canceled'` orders, the same way
- `total_orders` — `COUNT(*)`
- `completed_pct` — completed as a percentage of total, rounded to 2 decimal places
- `canceled_pct` — canceled as a percentage of total, rounded to 2 decimal places

There is a trap in the two percentage columns. `625 * 100 / 99441` is integer ÷ integer, and SQLite throws away everything after the decimal point — the answer comes back as a flat `0`, with no error to warn you. Force REAL division by multiplying one operand by `1.0`: `SUM(...) * 1.0 * 100 / COUNT(*)`, all wrapped in `ROUND(..., 2)`.

**Expected:** one row — 96,478 completed, 625 canceled, 99,441 total. The check cell computes the two expected percentages from those three verified counts, so if either share comes back as a whole number (or as `0`), you dropped the `* 1.0`.

In [10]:
%%sql q5 <<
-- Your query here
SELECT
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS completed_orders,
    SUM(CASE WHEN order_status = 'canceled'  THEN 1 ELSE 0 END) AS canceled_orders,
    COUNT(*) AS total_orders,
    ROUND(SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) * 1.0 * 100 / COUNT(*), 2) AS completed_pct,
    ROUND(SUM(CASE WHEN order_status = 'canceled'  THEN 1 ELSE 0 END) * 1.0 * 100 / COUNT(*), 2) AS canceled_pct
FROM orders;

In [11]:
# --- CHECK Q5 — do not edit ---
assert q5.shape[0] == 1, f"Q5: expected a single row, got {q5.shape[0]}"
for col in ['completed_orders', 'canceled_orders', 'total_orders', 'completed_pct', 'canceled_pct']:
    assert col in q5.columns, f"Q5: missing the '{col}' column — check your SELECT aliases"
for col, n in [('completed_orders', 96478), ('canceled_orders', 625), ('total_orders', 99441)]:
    assert int(q5.iloc[0][col]) == n, f"Q5: expected {col} = {n:,}, got {int(q5.iloc[0][col]):,}"

# Percentages are derived from the three verified counts above, not hard-coded.
for col, num in [('completed_pct', 96478), ('canceled_pct', 625)]:
    got, want = float(q5.iloc[0][col]), round(num * 100.0 / 99441, 2)
    assert abs(got - want) < 0.01, \
        (f"Q5: expected {col} ≈ {want} ({num:,} of 99,441), got {got} — "
         f"a whole number or 0 means integer division truncated it; multiply by 1.0")
print("✅ Q5 correct")
q5  # show the result of your query

✅ Q5 correct


,completed_orders,canceled_orders,total_orders,completed_pct,canceled_pct
0,96478,625,99441,97.02,0.63
